# MoLFormer on AWS Inferentia2 (inf2)

This notebook demonstrates how to compile and run [IBM MoLFormer-XL-both-10pct](https://huggingface.co/ibm/MoLFormer-XL-both-10pct) on AWS Inferentia2 using `torch_neuronx.trace()`.

MoLFormer is a ~47M parameter encoder-only molecular transformer that generates embeddings from SMILES (Simplified Molecular Input Line Entry System) strings. It is used in drug discovery, molecular property prediction, and cheminformatics.

**Instance**: inf2.xlarge (2 NeuronCores, SDK 2.28, PyTorch 2.9)

**Key results** (inf2.xlarge):
- Peak throughput: **1,660 inf/s** (BS=4, DataParallel, 2 cores)
- Single-core latency: **1.50 ms** P50 (BS=1, matmult bf16)
- Accuracy: cosine similarity **0.999995** vs CPU reference
- Model size: **74 MB** (matmult bf16) vs 160 MB (FP32) -- 54% smaller

## Step 0: Setup and Verification

In [ ]:
import subprocess
import json

# Verify Neuron environment
result = subprocess.run(['neuron-ls', '--json-output'], capture_output=True, text=True)
neuron_info = json.loads(result.stdout)
n_cores = sum(d['nc_count'] for d in neuron_info)
print(f'NeuronCores available: {n_cores}')

import torch
import torch_neuronx
print(f'PyTorch: {torch.__version__}')
print(f'torch_neuronx: {torch_neuronx.__version__}')

## Step 1: Load Model and Tokenizer

In [ ]:
import os
import time
import numpy as np
from transformers import AutoModel, AutoTokenizer

os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'True'

MODEL_ID = 'ibm/MoLFormer-XL-both-10pct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
cpu_model = AutoModel.from_pretrained(
    MODEL_ID,
    deterministic_eval=True,
    trust_remote_code=True,
)
cpu_model.eval()
print(f'Model parameters: {sum(p.numel() for p in cpu_model.parameters()):,}')

## Step 2: Prepare Inputs and Compile for Neuron

In [ ]:
def encode_smiles(tokenizer, smiles_list, max_length=202, batch_size=1):
    """Encode SMILES strings for MoLFormer input."""
    tokens = tokenizer(
        smiles_list,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt',
    )
    return (
        torch.repeat_interleave(tokens['input_ids'], batch_size, 0),
        torch.repeat_interleave(tokens['attention_mask'], batch_size, 0),
    )


# Example SMILES molecules
SMILES_EXAMPLES = [
    'Cn1c(=O)c2c(ncn2C)n(C)c1=O',  # caffeine
    'CC(=O)Oc1ccccc1C(=O)O',         # aspirin
    'CC(=O)NC1=CC=C(O)C=C1',          # acetaminophen
    'C1CCCCC1',                        # cyclohexane
]

# Trace input (single molecule, BS=1)
example_inputs = encode_smiles(tokenizer, [SMILES_EXAMPLES[0]], max_length=202, batch_size=1)
print(f'Input shape: {example_inputs[0].shape}')

In [ ]:
import shutil

SAVED_DIR = 'saved_models'
COMPILER_ARGS = ['--auto-cast', 'matmult', '--auto-cast-type', 'bf16']

os.makedirs(SAVED_DIR, exist_ok=True)

# Compile for batch sizes 1 and 4
for bs in [1, 4]:
    model_file = os.path.join(SAVED_DIR, f'molformer_bs{bs}.pt')
    if os.path.isfile(model_file):
        print(f'BS={bs}: Loading cached model from {model_file}')
        continue

    trace_inputs = encode_smiles(tokenizer, [SMILES_EXAMPLES[0]], max_length=202, batch_size=bs)
    workdir = f'workdir_molformer_bs{bs}'
    shutil.rmtree(workdir, ignore_errors=True)

    # Load fresh model for tracing
    trace_model = AutoModel.from_pretrained(
        MODEL_ID,
        deterministic_eval=True,
        trust_remote_code=True,
    )

    print(f'BS={bs}: Compiling...')
    start = time.time()
    neuron_model = torch_neuronx.trace(
        trace_model,
        trace_inputs,
        compiler_args=COMPILER_ARGS,
        compiler_workdir=workdir,
    )
    elapsed = time.time() - start
    neuron_model.save(model_file)
    size_mb = os.path.getsize(model_file) / (1024 * 1024)
    print(f'BS={bs}: Compiled in {elapsed:.1f}s, saved to {model_file} ({size_mb:.1f} MB)')

## Step 3: Accuracy Verification

In [ ]:
model_file = os.path.join(SAVED_DIR, 'molformer_bs1.pt')
neuron_model = torch.jit.load(model_file)

print('Verifying accuracy across multiple SMILES molecules...\n')

for smiles in SMILES_EXAMPLES:
    inputs = encode_smiles(tokenizer, [smiles], max_length=202, batch_size=1)

    with torch.no_grad():
        cpu_out = cpu_model(*inputs)
        neuron_out = neuron_model(*inputs)

    cpu_pooler = cpu_out.pooler_output
    neuron_pooler = neuron_out[1]  # traced model returns tuple: (last_hidden_state, pooler_output)

    max_diff = torch.max(torch.abs(cpu_pooler - neuron_pooler)).item()
    mean_diff = torch.mean(torch.abs(cpu_pooler - neuron_pooler)).item()
    cosine_sim = torch.nn.functional.cosine_similarity(
        cpu_pooler.flatten().unsqueeze(0),
        neuron_pooler.flatten().unsqueeze(0),
    ).item()

    print(f'SMILES: {smiles}')
    print(f'  Max diff: {max_diff:.6f}, Mean diff: {mean_diff:.6f}, Cosine sim: {cosine_sim:.6f}')
    print()

## Step 4: Benchmark -- Single Core

In [ ]:
import concurrent.futures

def benchmark(model, inputs, batch_size, n_models=1, n_workers=1, n_iters=100, warmup=10):
    """Run benchmark with configurable parallelism."""
    models = [model] if n_models == 1 else [torch.jit.load(model_file) for _ in range(n_models)]

    # Warmup
    for _ in range(warmup):
        for m in models:
            m(*inputs)

    latencies = []

    def worker(m):
        for _ in range(n_iters):
            start = time.time()
            m(*inputs)
            latencies.append(time.time() - start)

    begin = time.time()
    with concurrent.futures.ThreadPoolExecutor(max_workers=n_workers) as pool:
        futures = [pool.submit(worker, models[i % len(models)]) for i in range(n_workers)]
        for f in futures:
            f.result()
    duration = time.time() - begin

    inferences = len(latencies) * batch_size
    throughput = inferences / duration
    lat_ms = np.array(latencies) * 1000.0

    return {
        'throughput': throughput,
        'p50_ms': np.percentile(lat_ms, 50),
        'p90_ms': np.percentile(lat_ms, 90),
        'p99_ms': np.percentile(lat_ms, 99),
    }


# Single core benchmarks
model_bs1 = torch.jit.load(os.path.join(SAVED_DIR, 'molformer_bs1.pt'))
inputs_bs1 = encode_smiles(tokenizer, [SMILES_EXAMPLES[0]], max_length=202, batch_size=1)

result = benchmark(model_bs1, inputs_bs1, batch_size=1)
print(f'BS=1, 1 core: {result["throughput"]:.0f} inf/s, P50={result["p50_ms"]:.2f}ms, P99={result["p99_ms"]:.2f}ms')

## Step 5: Benchmark -- DataParallel (All NeuronCores)

In [ ]:
# DataParallel: 2 models on 2 NeuronCores
model_file = os.path.join(SAVED_DIR, 'molformer_bs1.pt')

# BS=1, DP=2
result_dp = benchmark(model_bs1, inputs_bs1, batch_size=1, n_models=2, n_workers=2)
print(f'BS=1, DP=2: {result_dp["throughput"]:.0f} inf/s, P50={result_dp["p50_ms"]:.2f}ms')

# BS=1, DP=2, oversubscribed (4 workers)
result_dp4 = benchmark(model_bs1, inputs_bs1, batch_size=1, n_models=2, n_workers=4)
print(f'BS=1, DP=2, 4w: {result_dp4["throughput"]:.0f} inf/s, P50={result_dp4["p50_ms"]:.2f}ms')

# BS=4, DP=2
model_bs4 = torch.jit.load(os.path.join(SAVED_DIR, 'molformer_bs4.pt'))
inputs_bs4 = encode_smiles(tokenizer, [SMILES_EXAMPLES[0]], max_length=202, batch_size=4)
result_bs4 = benchmark(model_bs4, inputs_bs4, batch_size=4, n_models=2, n_workers=2)
print(f'BS=4, DP=2: {result_bs4["throughput"]:.0f} inf/s, P50={result_bs4["p50_ms"]:.2f}ms')

## Results Summary

| Config | Throughput (inf/s) | P50 Latency (ms) |
|--------|-------------------:|------------------:|
| BS=1, 1 core | 661 | 1.50 |
| BS=1, DP=2 | 1,302 | 1.52 |
| BS=1, DP=2, 4 workers | 1,503 | 2.65 |
| **BS=4, DP=2** | **1,660** | 4.79 |

### Key Findings

1. **`--auto-cast matmult` is critical for FP32 models**: 65% throughput gain (400 -> 661 inf/s single-core)
2. **DataParallel scales near-perfectly**: 1.97x with 2 NeuronCores
3. **Cosine similarity 0.999995**: functionally identical embeddings with matmult bf16
4. **Model 54% smaller**: 74 MB (matmult) vs 160 MB (FP32)